### This notebook features a baseline XGBoost model.

In [13]:
import kagglehub
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [21]:
def load_raw_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.Dataframe]:
    """
    1. Fetch the eliptic dataset from kagglehub.
    2. Load the feature, edges, and classes CSVs into DataFrames.
    """
    path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
    print("Path to dataset files:", path)
    data_dir = Path(path) / "elliptic_bitcoin_dataset"

    features_path = data_dir / "elliptic_txs_features.csv"
    edges_path = data_dir / "elliptic_txs_edgelist.csv"
    classes_path = data_dir / "elliptic_txs_classes.csv"

    features = pd.read_csv(features_path, header=None)
    edges = pd.read_csv(edges_path)
    classes = pd.read_csv(classes_path)

    return features, edges, classes

In [15]:
def build_node_table(features: pd.DataFrame, classes: pd.Dataframe) -> pd.DataFrame:
    """
    Merge the features and classes table to create a node table with labels
    Column 0: Transaction id (txId)
    Column 1: time_step (1-49)
    Column 2-166: Remaining numeric features
    """
    
    # rename features
    features = features.copy()
    features.rename(columns={0: 'txId', 1: 'time_step'}, inplace=True)

    # Join with labels on transaction id
    nodes = features.merge(classes, on='txId', how="left")
    return nodes

In [16]:
def preprocess_data(nodes: pd.DataFrame) -> tuple(pd.DataFrame, pd.Series):
    """
    Prepare for supervised learning:
    Drop unlabeled transactions, split features and label, etc.
    """
    # Keep only labelled transactions
    labelled = nodes[nodes["class"] != "unknown"].copy()

    # Map string labels to ints for computation
    label_map = {"1" : 1, "2" : 0} # 1 = illicit, 0 = licit
    y = labelled["class"].map(label_map)

    # Save feature columns in X. Feature columns are all columns with integer labels (non-string)
    features = [c for c in labelled.columns if isinstance(c, int)]
    X = labelled[features].copy()

    return X,y


In [23]:
def train_xgboost(X: pd.DataFrame, y:pd.Series):
    """
    Train simple XGBoost baseline and print evaluation.

    ASIDE: Train/test split is random which doesn't account for time dependence.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y, 
        test_size = 0.2,
        stratify = y,
        random_state = 42
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200, 
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        tree_method="hist",
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("XGBoost Baseline Evaluation")
    print(classification_report(y_test, y_pred, target_names=["licit (0)", "illicit (1)"]))


In [24]:
def main():
    features, edges, classes = load_raw_data()

    nodes = build_node_table(features, classes)

    print("\nClass distribution (including 'unknown'):")
    print(nodes["class"].value_counts())

    X,y = preprocess_data(nodes)

    print("\nLabelled samples:", len(y))
    print("Class distribution (labelled only):")
    print(y.value_counts(normalize=True))


    train_xgboost(X,y)
main()

Path to dataset files: /Users/aditsingh/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1

Class distribution (including 'unknown'):
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Labelled samples: 46564
Class distribution (labelled only):
class
0    0.902392
1    0.097608
Name: proportion, dtype: float64
XGBoost Baseline Evaluation
              precision    recall  f1-score   support

   licit (0)       0.99      1.00      1.00      8404
 illicit (1)       1.00      0.91      0.95       909

    accuracy                           0.99      9313
   macro avg       0.99      0.96      0.97      9313
weighted avg       0.99      0.99      0.99      9313



### A supervised XGBoost baseline achieves high performance on the Elliptic dataset, which indicates that the engineered tabular features already capture sufficient information about a transaction. However, this baseline treats transactions as independent, ignores unlabeled nodes, and relies on static features, making it poorly suited for a real-time inference system. We therefore continue with a GNN not to maximize metrics, but to model transaction risk as a property of a graph, leveraging the unlabeled structure and enabling real-time (realistic) streaming detection.